In [ ]:
pip install -U scikit-learn

In [13]:
pip install nltk


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.7 MB/s eta 0:00:00


In [14]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import pickle
from win32com.client import Dispatch
import tkinter as tk

In [16]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\venug\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\venug\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [17]:
data = pd.read_csv('spam.csv', encoding="latin-1")

In [18]:
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [19]:
data.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')

In [20]:
data = data.loc[:,~data.columns.str.contains('^Unnamed')]

In [21]:
data.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [22]:
data['v1'] = data['v1'].map({'ham':0,'spam':1})

In [23]:
data.head()

,v1,v2
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [24]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [25]:
def preprocess_text(text):
    text = re.sub(r'\W', ' ', text)  
    text = text.lower()  
    text = text.split()  
    text = [lemmatizer.lemmatize(word) for word in text if word not in stop_words]
    return ' '.join(text)

In [26]:
data['v2'] = data['v2'].apply(preprocess_text)

In [27]:
tfidf = TfidfVectorizer(max_features=3000)
x = tfidf.fit_transform(data['v2']).toarray()
y = data['v1']

In [28]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [29]:
model = MultinomialNB()
model.fit(x_train, y_train)

MultinomialNB()

In [30]:
y_pred = model.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.979372197309417


In [31]:
pickle.dump(model, open('spam_model.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf_vectorizer.pkl', 'wb'))

In [32]:
#def speak(text):
#   speaker = Dispatch("SAPI.SpVoice")
#   speaker.Speak(text)

In [33]:
from win32com.client import Dispatch

from win32com.client import Dispatch

def speak(text):
    speaker = Dispatch("SAPI.SpVoice")
    
    voices = speaker.GetVoices()
    
    for voice in voices:
        if "Zira" in voice.GetDescription():  
            speaker.Voice = voice
            break
    
    speaker.Speak(text)



In [34]:
def result():
    message = text.get()
    vect = tfidf.transform([preprocess_text(message)]).toarray()
    prediction = model.predict(vect)
    if prediction[0] == 1:
        output_label.config(text="This is a Spam mail", bg='#FF4040')
        speak("This is a Spam mail")
        print("Prediction: Spam mail")  
    else:
        output_label.config(text="This is not a Spam mail", bg='lightgreen')
        speak("This is not a Spam mail")
        print("Prediction: Not a Spam mail") 

In [40]:
import tkinter as tk
from tkinter import Label
from PIL import Image,ImageTk
import requests
root = tk.Tk()
root.geometry("500x500")
root.configure(bg='lightblue')
from tkinter import Tk, Label
from PIL import Image, ImageTk
import requests

# Step 1: Download the image
image_url = "https://images.unsplash.com/photo-1543373014-cfe4f4bc1cdf?crop=entropy&cs=tinysrgb&fit=max&fm=jpg&ixid=MnwxMjA3fDB8MXxzZWFyY2h8Mnx8aGlnaCUyMHJlc29sdXRpb258fDB8fHx8MTYyNzE3Njc2MQ&ixlib=rb-1.2.1&q=80&w=1080"  # Replace with the actual image URL
image_filename = "background_image.jpg"

response = requests.get(image_url)
with open(image_filename, "wb") as file:
    file.write(response.content)

# Step 2: Create the Tkinter application
root = Tk()
root.title("Background Image Example")
root.geometry("800x600")  # Set the window size

# Step 3: Load the downloaded image
image = Image.open(image_filename)
bg_image = ImageTk.PhotoImage(image)

# Step 4: Add the image to a label and display it as a background
bg_label = Label(root, image=bg_image)
bg_label.place(x=0, y=0, relwidth=1, relheight=1)  # Stretch the image to cover the entire window

# Add a sample widget on top of the background
label = Label(root, text="Welcome to Tkinter!", font=("Arial", 24), bg="white")
label.place(relx=0.5, rely=0.5, anchor="center")  # Center the label

# Run the Tkinter main loop
root.mainloop()

frame = tk.Frame(root, bg='lightblue')
frame.pack(expand=True)

l2 = tk.Label(frame, text="Email Spam Classification Application", font=("Times New Roman", 16), bg='lightblue')
l2.pack(pady=5)

l1 = tk.Label(frame, text="Enter Your Message:", font=("Times New Roman", 14), bg='lightblue')
l1.pack(pady=5)

text = tk.Entry(frame, font=("Times New Roman", 14), bg='white')
text.pack(pady=5)

output_label = tk.Label(frame, text="", font=("Times New Roman", 16), bg='lightblue')
output_label.pack(pady=10)

B = tk.Button(frame, text="Check Spam", command=result, font=("Times New Roman", 12), bg='pink')
B.pack(pady=10)

root.mainloop()

TclError: image "pyimage2" doesn't exist

In [42]:
from tkinter import Tk, Label
from PIL import Image, ImageTk
import requests

# Step 1: Download the image
image_url = "https://images.unsplash.com/photo-1543373014-cfe4f4bc1cdf?crop=entropy&cs=tinysrgb&fit=max&fm=jpg&ixid=MnwxMjA3fDB8MXxzZWFyY2h8Mnx8aGlnaCUyMHJlc29sdXRpb258fDB8fHx8MTYyNzE3Njc2MQ&ixlib=rb-1.2.1&q=80&w=1080"  # Replace with the actual image URL
image_filename = "background_image.jpg"

response = requests.get(image_url)
with open(image_filename, "wb") as file:
    file.write(response.content)

# Step 2: Create the Tkinter application
root = Tk()
root.title("Background Image Example")
root.geometry("800x600")  # Set the window size

# Step 3: Load the downloaded image
image = Image.open(image_filename)
bg_image = ImageTk.PhotoImage(image)

# Step 4: Add the image to a label and display it as a background
bg_label = Label(root, image=bg_image)
bg_label.place(x=0, y=0, relwidth=1, relheight=1)  # Stretch the image to cover the entire window

# Add a sample widget on top of the background
label = Label(root, text="Welcome to Tkinter!", font=("Arial", 24), bg="white")
label.place(relx=0.5, rely=0.5, anchor="center")  # Center the label

# Run the Tkinter main loop
root.mainloop()


TclError: image "pyimage3" doesn't exist